In [31]:
import json
import numpy as np
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import os

In [32]:
base_path = 'saves/eval'

In [33]:
runs = list(os.listdir(base_path))

In [34]:
runs

['mia_gutenberg_popular_eval_hubble-1b-500b_toks-perturbed-hf',
 'mia_gutenberg_popular_eval_hubble-1b-500b_toks-perturbed-hf_dup_1',
 'mia_gutenberg_popular_eval_hubble-1b-500b_toks-perturbed-hf_dup_16',
 'mia_gutenberg_popular_eval_hubble-1b-500b_toks-perturbed-hf_dup_256',
 'mia_gutenberg_popular_eval_hubble-1b-500b_toks-standard-hf',
 'mia_gutenberg_popular_eval_hubble-1b-500b_toks-standard-hf_dup_1',
 'mia_gutenberg_popular_eval_hubble-1b-500b_toks-standard-hf_dup_16',
 'mia_gutenberg_popular_eval_hubble-1b-500b_toks-standard-hf_dup_256',
 'mia_gutenberg_popular_eval_hubble-8b-500b_toks-perturbed-hf',
 'mia_gutenberg_popular_eval_hubble-8b-500b_toks-perturbed-hf_dup_1',
 'mia_gutenberg_popular_eval_hubble-8b-500b_toks-perturbed-hf_dup_16',
 'mia_gutenberg_popular_eval_hubble-8b-500b_toks-perturbed-hf_dup_256',
 'mia_gutenberg_popular_eval_hubble-8b-500b_toks-standard-hf',
 'mia_gutenberg_popular_eval_hubble-8b-500b_toks-standard-hf_dup_1',
 'mia_gutenberg_popular_eval_hubble-8b-50

# Per Scenario Plot

In [ ]:
for run in runs:

    if os.path.exists(f'plots/roc_auc_curves/{run}.pdf'):
        print(f"Skipping existing run: {run}")
        continue

    print(f"Processing run: {run}")
    # Load your evaluation results
    with open(f'saves/eval/{run}/TOFU_EVAL.json', 'r') as f:
        eval_results = json.load(f)

    plt.figure(figsize=(6, 6))

    metrics_to_plot = ['mia_loss', 'mia_min_k', 'mia_min_k_plus_plus', 'mia_zlib']

    metric_names_formatted = {
        'mia_loss': 'Loss',
        'mia_min_k': 'MinK%',
        'mia_min_k_plus_plus': 'MinK%++',
        'mia_zlib': 'ZLib'
    }

    data = {}

    for metric_name in metrics_to_plot:
        if metric_name in eval_results:
            metric_data = eval_results[metric_name]

            # Extract membership scores
            forget_scores = np.array([
                elem["score"] for elem in metric_data["forget"]["value_by_index"].values()
            ])
            holdout_scores = np.array([
                elem["score"] for elem in metric_data["holdout"]["value_by_index"].values()
            ])

            # Labels
            y_true = np.concatenate([np.ones(len(holdout_scores)),
                                    np.zeros(len(forget_scores))])
            y_scores = np.concatenate([holdout_scores, forget_scores])

            # ROC
            fpr, tpr, _ = roc_curve(y_true, y_scores)
            roc_auc = auc(fpr, tpr)

            # TPR @ 1% FPR (using interpolation)
            tpr_at_1pct_fpr = np.interp(0.01, fpr, tpr)
            tpr_at_1pct_fpr_as_percentage = tpr_at_1pct_fpr * 100

            formatted_name = metric_names_formatted.get(metric_name, metric_name)

            data[formatted_name] = {
                "fpr": list(fpr),
                "tpr": list(tpr),
                "roc_auc": roc_auc,
                "tpr_at_1pct_fpr": tpr_at_1pct_fpr,
                "tpr_at_1pct_fpr_as_percentage": tpr_at_1pct_fpr_as_percentage
            }

            print(f"{formatted_name}:")
            print(f"  ROC AUC: {roc_auc:.4f}")
            print(f"  TPR @ 1% FPR: {tpr_at_1pct_fpr:.4f}")

            label = f"{formatted_name} (AUC={roc_auc:.3f}, TPR@1%FPR={tpr_at_1pct_fpr:.3f})"
            plt.plot(fpr, tpr, label=label)

    # Final plot formatting
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.title('ROC Curves for All MIA Metrics')
    plt.legend(fontsize=10)
    plt.grid(alpha=0.3)

    # label size
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # plt.show()

    # Save the plot as PDF
    plt.savefig(f'plots/roc_auc_curves/{run}.pdf', bbox_inches='tight')
    plt.close()

    with open(f'plots/roc_auc_curves/{run}_mia_results.json', 'w') as f:
        json.dump(data, f, indent=4)

Skipping existing run: mia_gutenberg_popular_eval_hubble-1b-500b_toks-perturbed-hf
Skipping existing run: mia_gutenberg_popular_eval_hubble-1b-500b_toks-perturbed-hf_dup_1
Skipping existing run: mia_gutenberg_popular_eval_hubble-1b-500b_toks-perturbed-hf_dup_16
Skipping existing run: mia_gutenberg_popular_eval_hubble-1b-500b_toks-perturbed-hf_dup_256
Skipping existing run: mia_gutenberg_popular_eval_hubble-1b-500b_toks-standard-hf
Skipping existing run: mia_gutenberg_popular_eval_hubble-1b-500b_toks-standard-hf_dup_1
Skipping existing run: mia_gutenberg_popular_eval_hubble-1b-500b_toks-standard-hf_dup_16
Skipping existing run: mia_gutenberg_popular_eval_hubble-1b-500b_toks-standard-hf_dup_256
Skipping existing run: mia_gutenberg_popular_eval_hubble-8b-500b_toks-perturbed-hf
Skipping existing run: mia_gutenberg_popular_eval_hubble-8b-500b_toks-perturbed-hf_dup_1
Skipping existing run: mia_gutenberg_popular_eval_hubble-8b-500b_toks-perturbed-hf_dup_16
Skipping existing run: mia_gutenberg

# Plot with varying Dup counts

In [49]:
tasks = [
    'gutenberg_popular', 
    'gutenberg_unpopular', 
    'yago_biographies', 
    'mmlu', 
    'passage_wikipedia'
]

models = ['hubble-8b-500b_toks', 'hubble-1b-500b_toks']

suffixes = ['standard', 'perturbed']

duplicates = [0, 1, 4, 16, 64, 256]

metrics_to_plot = ['mia_loss', 'mia_min_k', 'mia_min_k_plus_plus', 'mia_zlib']

metric_names_formatted = {
    'mia_loss': 'Loss',
    'mia_min_k': 'MinK%',
    'mia_min_k_plus_plus': 'MinK%++',
    'mia_zlib': 'ZLib'
}

for task in tasks:
    for model in models:
        for suffix in suffixes:
            for dup in duplicates:
                
                if task == 'gutenberg_popular' and dup in [4, 64]:
                    print("Skipping invalid configuration.")
                    continue

                run = f"mia_{task}_eval_{model}-{suffix}-hf"
                if dup > 0:
                    run += f"_dup_{dup}"
                
                with open(f'saves/eval/{run}/TOFU_EVAL.json', 'r') as f:
                    eval_results = json.load(f)

Skipping invalid configuration.
Skipping invalid configuration.
Skipping invalid configuration.
Skipping invalid configuration.
Skipping invalid configuration.
Skipping invalid configuration.
Skipping invalid configuration.
Skipping invalid configuration.


In [54]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

tasks = [
    'gutenberg_popular', 
    'gutenberg_unpopular', 
    'yago_biographies', 
    'mmlu', 
    'passage_wikipedia'
]

models = ['hubble-8b-500b_toks', 'hubble-1b-500b_toks']
suffixes = ['standard', 'perturbed']

duplicates = [0, 1, 4, 16, 64, 256]

metrics_to_plot = ['mia_loss', 'mia_min_k', 'mia_min_k_plus_plus', 'mia_zlib']

metric_names_formatted = {
    'mia_loss': 'Loss',
    'mia_min_k': 'MinK%',
    'mia_min_k_plus_plus': 'MinK%++',
    'mia_zlib': 'ZLib'
}

output_dir = "plots/roc_grid"
os.makedirs(output_dir, exist_ok=True)

for task in tasks:
    for model in models:
        for suffix in suffixes:

            print(f"Creating 2x2 ROC grid → task={task}, model={model}, suffix={suffix}")

            plt.figure(figsize=(12, 12))
            fig, axes = plt.subplots(2, 2, figsize=(12, 12))
            axes = axes.flatten()

            # For legend consistency across subplots
            dup_labels_added = set()

            for metric_idx, metric_name in enumerate(metrics_to_plot):
                ax = axes[metric_idx]
                formatted_metric = metric_names_formatted.get(metric_name, metric_name)

                for dup in duplicates:

                    # Skip invalid combinations
                    if task == 'gutenberg_popular' and dup in [4, 64]:
                        print(f"  Skipping invalid run for dup={dup}")
                        continue

                    run = f"mia_{task}_eval_{model}-{suffix}-hf"
                    if dup > 0:
                        run += f"_dup_{dup}"

                    json_path = f"saves/eval/{run}/TOFU_EVAL.json"
                    if not os.path.exists(json_path):
                        print(f"  Missing file → {json_path} (skipping)")
                        continue

                    with open(json_path, "r") as f:
                        eval_results = json.load(f)

                    if metric_name not in eval_results:
                        print(f"  Metric missing → {metric_name} in run {run}")
                        continue

                    metric_data = eval_results[metric_name]

                    forget_scores = np.array([
                        v["score"] for v in metric_data["forget"]["value_by_index"].values()
                    ])
                    holdout_scores = np.array([
                        v["score"] for v in metric_data["holdout"]["value_by_index"].values()
                    ])

                    y_true = np.concatenate([
                        np.ones(len(holdout_scores)),
                        np.zeros(len(forget_scores))
                    ])
                    y_scores = np.concatenate([holdout_scores, forget_scores])

                    fpr, tpr, thresholds = roc_curve(y_true, y_scores)

                    # Interpolation for TPR@1% FPR
                    tpr_at_1pct_fpr = np.interp(0.01, fpr, tpr)
                    roc_auc_val = auc(fpr, tpr)

                    label = f"Unseen v/s Dup={dup} (AUC={roc_auc_val:.3f}, TPR@1%FPR={tpr_at_1pct_fpr:.3f})" if dup != 0 else f"Unseen v/s Seen (AUC={roc_auc_val:.3f}, TPR@1%FPR={tpr_at_1pct_fpr:.3f})"

                    ax.plot(fpr, tpr, label=label)

                ax.set_title(f"{formatted_metric}", fontsize=16)
                ax.set_xlabel("False Positive Rate", fontsize=12)
                ax.set_ylabel("True Positive Rate", fontsize=12)
                ax.grid(alpha=0.3)
                ax.legend(fontsize=8)

            # plt.suptitle(
            #     f"ROC Curves for {task} — {model} — {suffix}", 
            #     fontsize=20
            # )

            plt.tight_layout(rect=[0, 0, 1, 0.97])

            output_path = f"{output_dir}/{task}_{model}_{suffix}_roc_grid.pdf"
            plt.savefig(output_path)
            plt.close()

            print(f"Saved → {output_path}")


Creating 2x2 ROC grid → task=gutenberg_popular, model=hubble-8b-500b_toks, suffix=standard
  Skipping invalid run for dup=4
  Skipping invalid run for dup=64
  Skipping invalid run for dup=4
  Skipping invalid run for dup=64
  Skipping invalid run for dup=4
  Skipping invalid run for dup=64
  Skipping invalid run for dup=4
  Skipping invalid run for dup=64
Saved → plots/roc_grid/gutenberg_popular_hubble-8b-500b_toks_standard_roc_grid.pdf
Creating 2x2 ROC grid → task=gutenberg_popular, model=hubble-8b-500b_toks, suffix=perturbed
  Skipping invalid run for dup=4
  Skipping invalid run for dup=64
  Skipping invalid run for dup=4
  Skipping invalid run for dup=64
  Skipping invalid run for dup=4
  Skipping invalid run for dup=64
  Skipping invalid run for dup=4
  Skipping invalid run for dup=64
Saved → plots/roc_grid/gutenberg_popular_hubble-8b-500b_toks_perturbed_roc_grid.pdf
Creating 2x2 ROC grid → task=gutenberg_popular, model=hubble-1b-500b_toks, suffix=standard
  Skipping invalid run 

C:\Users\aflah\AppData\Local\Temp\ipykernel_11660\41043491.py:39: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(2, 2, figsize=(12, 12))


Saved → plots/roc_grid/passage_wikipedia_hubble-1b-500b_toks_perturbed_roc_grid.pdf


<Figure size 1200x1200 with 0 Axes>

<Figure size 1200x1200 with 0 Axes>

...

<Figure size 1200x1200 with 0 Axes>

<Figure size 1200x1200 with 0 Axes>

<Figure size 1200x1200 with 0 Axes>

<Figure size 1200x1200 with 0 Axes>

<Figure size 1200x1200 with 0 Axes>